# Notebook 03: Mrezne mjere kao prediktori volatilnosti

**Projekt:** Financijske mreze na ZSE (CROBEX, 2004-2026)
**Teme:** Spearmanov rho, kondicionalna analiza, scatter plotovi, prediktivnost
**Kolegiji:** Strojno ucenje u financijama, Financijsko modeliranje

## Kontekst

Pitanje: **mogu li mrezne mjere predvidjeti volatilnost trzista u sljedecem periodu?**

Koristimo **Spearmanov koeficijent korelacije ranga** (rho) jer:
- ne pretpostavlja normalnost
- robustan na ekstremne vrijednosti
- standardni alat u empirijskim financijama za slabe monotone veze

Kljucna razlika od Notebook 02: tamo smo gledali *trenutni* stres (je li sada kriza?).
Ovdje gledamo *predskazivanje* — mjera u periodu t predvidja volatilnost u periodu t+1.


In [ ]:
# Postavljanje okolisa (automatski detektira Google Colab)
import sys
if "google.colab" in sys.modules:
    import urllib.request, os
    os.makedirs("/content/sample_data", exist_ok=True)
    _base = "https://raw.githubusercontent.com/svlah-sketch/FinNet-teaching/main/sample_data/"
    for _f in ["sample_metrics_W90.csv", "sample_metrics_revision.csv",
               "CROBEX_values.csv", "Revisions.csv"]:
        urllib.request.urlretrieve(_base + _f, f"/content/sample_data/{_f}")
    print("Colab: podaci preuzeti s GitHuba.")
else:
    print("Lokalno pokretanje.")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy import stats
from pathlib import Path

# Automatski pronalazi sample_data/ (Colab ili lokalno)
import sys
if "google.colab" in sys.modules:
    DATA_DIR = Path("/content/sample_data")
else:
    for _p in [Path("../sample_data"), Path("sample_data")]:
        if _p.exists() and list(_p.glob("*.csv")):
            DATA_DIR = _p
            break
    else:
        raise FileNotFoundError("sample_data/ nije pronaden. Pokreni prvu celiju notebooka.")
print(f"DATA_DIR: {DATA_DIR.resolve()}")

# Ucitaj CROBEX dnevne log-prinose
crobex = pd.read_csv(DATA_DIR / "CROBEX_values.csv", sep=";", encoding="cp1250")
crobex["date"] = pd.to_datetime(crobex["date"], dayfirst=True)
crobex["crobex"] = pd.to_numeric(crobex["crobex"], errors="coerce")
crobex_ret = np.log(crobex.sort_values("date").set_index("date")["crobex"]).diff().dropna()

# Ucitaj revizijske prozore
revs = pd.read_csv(DATA_DIR / "Revisions.csv", sep=";", encoding="cp1250")
revs["start_date"] = pd.to_datetime(revs["start_date"], dayfirst=True)
revs["end_date"]   = pd.to_datetime(revs["end_date"],   dayfirst=True)
revs = revs[(revs["rev_cnt"] >= 10) & (revs["rev_cnt"] <= 52)].reset_index(drop=True)

# Ucitaj mrezne mjere (revizijski prozori)
metrics = pd.read_csv(DATA_DIR / "sample_metrics_revision.csv",
                      index_col="window_end", parse_dates=True)

print(f"CROBEX: {len(crobex_ret)} dnevnih prinosa")
print(f"Revizijskih prozora: {len(revs)}")
print(f"Mreznih mjera: {metrics.shape}")


## 1. Realizirana volatilnost po prozoru

Za svaki revizijski prozor izracunavamo anualiziranu standardnu devijaciju dnevnih log-prinosa.
Cilj prognoze je **vol_{t+1}** — volatilnost *sljedeceg* prozora.

$$\sigma_t = \text{std}(r_d,\, d \in [t_{start}, t_{end}]) \times \sqrt{252}$$


In [ ]:
# Izracunaj realiziranu volatilnost po prozoru
rows = []
for _, row in revs.iterrows():
    r = crobex_ret.loc[row["start_date"]:row["end_date"]].dropna()
    vol = float(r.std(ddof=1) * np.sqrt(252)) if len(r) >= 10 else np.nan
    diffs = (metrics.index - row["end_date"]).to_series().abs()
    best  = int(diffs.values.argmin())
    met_row = metrics.iloc[best] if diffs.iloc[best].days <= 5 else pd.Series(dtype=float)
    entry = {"end_date": row["end_date"], "start_date": row["start_date"], "vol": vol}
    for col in metrics.columns:
        entry[col] = float(met_row[col]) if col in met_row.index else np.nan
    rows.append(entry)

df = pd.DataFrame(rows)
df["vol_next"] = df["vol"].shift(-1)  # volatilnost SLJEDECEG prozora
df = df.iloc[:-1].copy()              # izbaci zadnji (nema vol_next)

print(f"Prozora za analizu: {df["vol_next"].notna().sum()}")
print(df[["end_date","vol","vol_next","M1_LCC","M4_NCom"]].head(6).to_string(index=False))


**Sto vidimo u tablici:** Svaki redak = jedan revizijski prozor.
`vol` = realizirana volatilnost tog prozora; `vol_next` = volatilnost *sljedeceg* prozora
(to je ono sto pokusavamo predvidjeti). Mrezne mjere (M1_LCC, M4_NCom...) su vrijednosti
iz tog istog prozora — koristimo ih kao prediktore za `vol_next`.

> M4_NCom je NaN u vecini prozora — to je ocekivano (mjera je kondicionalna).


## 2. Spearmanov rho: koje mjere predvidjaju volatilnost?

Za svaku od 10 mjera racunamo Spearman korelaciju s **vol_next** (buducom volatilnoscu).

**Ocekivani smjerovi** (iz teorije i istrazivakog nalaza):
- M1, M2, M3, M6, M7: pozitivan rho (visi stres -> visa buduća volatilnost)
- M8, M9: negativan rho (visi hub = nizi stres -> niza volatilnost)
- M4, M5: pozitivan rho, ali **kondicionalno** (samo u stresnim prozorima)


In [ ]:
# Spearmanov ro za svaku mjeru vs. sljedeca volatilnost
results = []
for col in metrics.columns:
    x = df[col].values
    y = df["vol_next"].values
    mask = ~np.isnan(x) & ~np.isnan(y)
    n = mask.sum()
    if n < 5:
        continue
    rho, pval = stats.spearmanr(x[mask], y[mask])
    results.append({"mjera": col, "n": n, "rho": round(rho,3), "p": round(pval,4),
                    "sig": "**" if pval<0.01 else ("*" if pval<0.05 else
                           ("." if pval<0.10 else ""))})

res = pd.DataFrame(results).sort_values("rho", key=abs, ascending=False)
print("Spearmanov ro: mrezne mjere vs. buduća volatilnost (revizijski prozori)")
print(res[["mjera","n","rho","p","sig"]].to_string(index=False))
print()
print("** p<0.01  * p<0.05  . p<0.10")
print()
print("Napomena: M4, M8 imaju mali n jer su dostupne samo u stresnim periodima")
print("(NG-0.001 neprazna). To je kondicionalna statistika.")


**Kako citati tablicu:**

- `rho` = Spearmanov koeficijent korelacije ranga. Raspon [-1, +1].
  Pozitivan: visa mjera danas → visa volatilnost sutra. Negativan: obrnuto.
- `n` = broj prozora gdje mjera nije NaN. Za M4, M5, M8 ocekujemo mali n
  (~15-18) jer su kondicionalne mjere.
- `sig` = statisticka znacajnost: `**` p<0.01, `*` p<0.05, `.` p<0.10

**Sto ocekivati:**
Mjere poput M1 (LCC), M3 (stupanj) i M6 (apsorcijski omjer) trebale bi imati
pozitivan rho — veca gustoca NG mreze danas predvidja nestabilnost sutra.
M4 (broj zajednica) ima mali n ali snazno pozitivan rho — kljucni nalaz istrazivanja.


## 3. Scatter plotovi: mjera vs. buduća volatilnost

Svaka tocka = jedan revizijski prozor. **Crveno = krizni period**, plavo = mirno.


In [ ]:
# Scatter plotovi: 4 kljucne mjere vs. sljedeca volatilnost
CRISES = {
    "GFC":     ("2007-10-01", "2009-09-30"),
    "EU_DEBT": ("2011-04-01", "2012-09-30"),
    "COVID":   ("2020-02-20", "2021-03-31"),
}

def is_crisis(date):
    for s, e in CRISES.values():
        if pd.Timestamp(s) <= date <= pd.Timestamp(e):
            return True
    return False

df["kriza"] = df["end_date"].map(is_crisis)

PLOT_COLS = ["M1_LCC", "M3_MeanDeg", "M4_NCom", "M6_AbsRat"]
fig, axes = plt.subplots(2, 2, figsize=(13, 10))

for ax, col in zip(axes.flat, PLOT_COLS):
    mask = df[col].notna() & df["vol_next"].notna()
    x_c = df.loc[mask & df["kriza"],  col]
    y_c = df.loc[mask & df["kriza"],  "vol_next"]
    x_t = df.loc[mask & ~df["kriza"], col]
    y_t = df.loc[mask & ~df["kriza"], "vol_next"]
    ax.scatter(x_t, y_t, color="steelblue", alpha=0.7, s=50, label="Mirno")
    ax.scatter(x_c, y_c, color="crimson",   alpha=0.8, s=70, label="Kriza")
    rho, p = stats.spearmanr(df.loc[mask, col], df.loc[mask, "vol_next"])
    n = mask.sum()
    ax.set_xlabel(col, fontsize=9)
    ax.set_ylabel("Buduća vol. (anualizirana)", fontsize=9)
    ax.set_title(f"{col}\nSpearman rho={rho:+.3f}, p={p:.3f}, n={n}", fontsize=10)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

fig.suptitle("Mrezne mjere vs. buduća volatilnost\n"
             "(crveno = krizni prozor, plavo = mirni prozor)", fontsize=12)
plt.tight_layout()
plt.savefig("nb03_scatter.png", dpi=120, bbox_inches="tight")
plt.show()


**Kako citati scatter plotove:**

- Svaka tocka = jedan revizijski prozor (2004-2025)
- Os x = vrijednost mrezne mjere na kraju tog prozora
- Os y = realizirana volatilnost CROBEX-a u *sljedecem* prozoru
- **Crvene tocke** = prozori ciji kraj pada u krizno razdoblje (GFC, EU dug, COVID)
- **Plave tocke** = mirni prozori

Pozitivan rho → oblak tocaka ide "gore desno" (vise mjere → vise vol_next).
Ako su crvene tocke grupirane gore desno → krize daju jaci signal.

> Napomena: M4 ima malo tocaka (kondicionalna mjera) — tocke koje vidite
> su samo stresni prozori gdje NG-0.001 nije bila prazna.


## 4. Kondicionalna analiza: zasto M4 radi samo u stresnim periodima

**M4 (broj zajednica u NG-0.001)** je posebna mjera:
- Dostupna samo kada je NG-0.001 mreza neprazna (~30-40% prozora)
- Ovi "aktivni" prozori su upravo stresni periodi
- Mjera dakle vec po definiciji selektira stresne epizode

Usporedujemo: rho u **svim stresnim prozorima** vs. samo **kriznim** vs. samo **mirnim stresnim**.


In [ ]:
# Kondicionalna analiza: M4 u svim vs. samo stresnim prozorima
# M4 (broj zajednica u NG-0.001) aktivan samo kad je mreza pod stresom

mask_all    = df["M4_NCom"].notna() & df["vol_next"].notna()
mask_crisis = mask_all & df["kriza"]
mask_tran   = mask_all & ~df["kriza"]

def rho_n(x, y):
    r, p = stats.spearmanr(x, y)
    return r, p, len(x)

r_all,  p_all,  n_all  = rho_n(df.loc[mask_all,    "M4_NCom"], df.loc[mask_all,    "vol_next"])
r_cris, p_cris, n_cris = rho_n(df.loc[mask_crisis, "M4_NCom"], df.loc[mask_crisis, "vol_next"])
r_tran, p_tran, n_tran = rho_n(df.loc[mask_tran,   "M4_NCom"], df.loc[mask_tran,   "vol_next"]) if mask_tran.sum()>=3 else (np.nan, np.nan, 0)

print("M4 (broj zajednica u NG-0.001) — kondicionalna analiza:")
print(f"  Svi stresni prozori:  rho={r_all:+.3f}  p={p_all:.4f}  n={n_all}")
print(f"  Samo krizni:          rho={r_cris:+.3f}  p={p_cris:.4f}  n={n_cris}")
print(f"  Samo mirni (stres):   rho={r_tran:+.3f}  p={p_tran:.4f}  n={n_tran}")
print()

# Vizualizacija: scatter M4 vs vol_next, bojano po krizi
fig, ax = plt.subplots(figsize=(8, 5))
for label, mask_s, color in [
    ("Stresno + krizno", mask_crisis, "crimson"),
    ("Stresno + mirno",  mask_tran,   "steelblue"),
]:
    if mask_s.sum() > 0:
        ax.scatter(df.loc[mask_s, "M4_NCom"], df.loc[mask_s, "vol_next"],
                   label=f"{label} (n={mask_s.sum()})", color=color, s=70, alpha=0.8)

ax.set_xlabel("M4: Broj zajednica (NG-0.001, kondicionalno)", fontsize=10)
ax.set_ylabel("Buduća volatilnost (anualizirana)", fontsize=10)
ax.set_title(f"M4 vs. buduća vol.\n"
             f"Svi stresni: rho={r_all:+.3f} (p={p_all:.3f}, n={n_all})", fontsize=11)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("nb03_m4_cond.png", dpi=120, bbox_inches="tight")
plt.show()

print("Zakljucak: M4 je visoko koreliran s buducom volatilnoscu u stresnim periodima.")
print("U mirnim periodima NG-0.001 mreza je prazna -> M4 = NaN -> nije dostupno.")
print("Ovo je kondicionalna, ne bezuvjetna statistika.")


**Kako citati output kondicionalne analize:**

- **Svi stresni prozori** = svi prozori gdje je M4 dostupan (NG-0.001 aktivna)
- **Samo krizni** = od tih stresnih, oni ciji kraj pada u GFC/EU dug/COVID
- **Samo mirni (stres)** = stresni prozori izvan definiranih kriza
  (moze biti malo ili nula ako stres po NG-0.001 uvijek pada u krize)

Visok rho u kriznim + dobar n → M4 je pouzdan signal u kriznim epizodama.
Ako je n za mirne stresne prozore mali (<5), taj rho treba uzeti s oprezom.

**Scatter plot:** Tocke su obojene je li prozor krizni ili mirni-stresni.
Ocekujemo pozitivan nagib — vise zajednica danas → veca volatilnost sutra.


## 5. Out-of-sample (OOS) validacija: NET mreza vs. GARCH(1,1)

Spearman rho je **in-sample** mjera — govori nam da korelacija postoji,
ali ne provjerava prognoznu moc izvan uzorka.

OOS test: za svaki prozor $t$ (od 15. nadalje) model se trenira samo na
podacima $1, ..., t-1$ (**expanding window**) i prognozira volatilnost prozora $t+1$.
Nikad ne "vidi" buduce podatke.

**Baseline:** GARCH(1,1) — standardni model u praksi i literaturi.
**Alternativa:** NET model — OLS s mreznom mjerom kao jedinim prediktorom.

**QLIKE gubitak** (Patton, 2011): $\text{QLIKE} = \log(\hat{\sigma}^2) + \sigma^2_{real}/\hat{\sigma}^2$
— nizi je bolji, asimetricno penalizira podcjenjivanje rizika.

**DM test** (Harvey, Leybourne, Newbold, 1997): $H_0$ = jednaka prognozna tocnost.
Negativan DM statistik -> NET ima nizi QLIKE -> NET bolji od GARCH.

> *Napomena: GARCH fitting traje ~1-2 minute u Colabu.*


In [ ]:
# OOS usporedba: NET (M1, M2) vs GARCH(1,1)
# Expanding window: model se trenira samo na proslim podacima
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "arch"], check=True)
import warnings
from numpy.linalg import lstsq
from arch import arch_model

MIN_TRAIN = 15
N = len(df)
garch_hat    = np.full(N, np.nan)
net_m1_hat   = np.full(N, np.nan)
net_m2_hat   = np.full(N, np.nan)
hist_mean_hat = np.full(N, np.nan)  # historical mean
last_obs_hat  = np.full(N, np.nan)  # random walk (last observed)

print("Fitam GARCH i OLS modele (moze trajati ~1-2 min)...")
for t in range(MIN_TRAIN, N):
    if t + 1 >= N:
        continue
    end_t  = df["end_date"].iloc[t]
    s_next = df["start_date"].iloc[t+1]
    e_next = df["end_date"].iloc[t+1]
    r_next = crobex_ret.loc[s_next:e_next].dropna()
    h = max(len(r_next), 1)
    # GARCH(1,1)
    train = crobex_ret.loc[:end_t].dropna()
    if len(train) >= 100:
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                am  = arch_model(train*100, vol="Garch", p=1, q=1,
                                 mean="Constant", dist="normal")
                res = am.fit(disp="off", show_warning=False)
                fc  = res.forecast(horizon=h, reindex=False)
                garch_hat[t] = np.sqrt(np.mean(fc.variance.values[-1]/1e4)) * np.sqrt(252)
        except Exception:
            pass
    # OLS expanding (NET M1, NET M2)
    y   = df["vol_next"].iloc[:t].values
    for col, arr in [("M1_LCC", net_m1_hat), ("M2_APL", net_m2_hat)]:
        x = df[col].iloc[:t].values
        ok = ~np.isnan(y) & ~np.isnan(x)
        if ok.sum() >= 5:
            X = np.column_stack([np.ones(ok.sum()), x[ok]])
            coef, *_ = lstsq(X, y[ok], rcond=None)
            pred = coef[0] + coef[1]*df[col].iloc[t]
            arr[t] = max(float(pred), 0.01) if not np.isnan(pred) else np.nan

    # Jednostavni benchmarci (ne trebaju fit)
    y_past = df["vol_next"].iloc[:t].dropna()
    hist_mean_hat[t] = float(y_past.mean()) if len(y_past) >= 3 else np.nan
    last_obs_hat[t]  = float(df["vol"].iloc[t])  # random walk: vol(t) predvida vol(t+1)

df["garch_hat"]     = garch_hat
df["net_m1_hat"]    = net_m1_hat
df["net_m2_hat"]    = net_m2_hat
df["hist_mean_hat"] = hist_mean_hat
df["last_obs_hat"]  = last_obs_hat
n_oos = int((~np.isnan(garch_hat)).sum())
print(f"OOS prozora (GARCH dostupan): {n_oos}")


In [ ]:
# QLIKE gubitak i DM test
from scipy import stats as scipy_stats

def qlike(hat, real):
    h2 = hat**2
    return np.log(h2) + real**2 / h2

def dm_hlm(loss_alt, loss_bench):
    """Harvey-Leybourne-Newbold DM test. Negativan stat = alt bolji."""
    d = loss_alt - loss_bench
    T = len(d)
    dm = d.mean() / np.sqrt(np.var(d, ddof=1) / T)
    dm_adj = dm * np.sqrt((T + 1 - 2 + 1.0/T) / T)
    p = 2 * (1 - scipy_stats.norm.cdf(abs(dm_adj)))
    return float(dm_adj), float(p)

# OOS skup: samo prozori gdje GARCH ima prognozu
y_r = df["vol_next"].values
oos_mask = ~np.isnan(garch_hat) & ~np.isnan(y_r)
ql_garch = qlike(garch_hat[oos_mask], y_r[oos_mask])

hdr = f"{"Model":<12} {"Mean QLIKE":>12} {"DM stat":>9} {"p":>8} {"Bolji od GARCH?":>16}"
print(hdr)
print("-" * 62)
g_line = f"{"GARCH(1,1)":<12} {np.mean(ql_garch):>12.4f}"
print(g_line)
for label, hat in [("Hist. mean", hist_mean_hat), ("Last obs.", last_obs_hat),
                    ("NET M1", net_m1_hat), ("NET M2", net_m2_hat)]:
    mask = oos_mask & ~np.isnan(hat)
    if mask.sum() < 5:
        continue
    ql = qlike(hat[mask], y_r[mask])
    ql_g = qlike(garch_hat[mask], y_r[mask])
    stat, p = dm_hlm(ql, ql_g)
    bolji = "DA **" if stat < 0 and p < 0.05 else (
            "DA ." if stat < 0 and p < 0.10 else (
            "-" if p >= 0.10 else "LOSIJI"))
    print(f"{label:<12} {np.mean(ql):>12.4f} {stat:>9.3f} {p:>8.4f} {bolji:>16}")
print()
print("** p<0.05  . p<0.10  - nema znacajne razlike")
print("Negativan DM stat -> NET ima nizi QLIKE -> bolji prediktor od GARCH")


**Kako citati OOS tablicu:**

- `Mean QLIKE` = prosjecni gubitak modela kroz sve OOS prozore. **Nizi = bolji.**
- `DM stat` = Diebold-Mariano statistik (usporedba s GARCH).
  **Negativan** → NET ima nizi QLIKE → **NET je bolji od GARCH**.
  Pozitivan → NET je gori.
- `p` = p-vrijednost DM testa. Mali p (< 0.10) → razlika je statisticki znacajna.
- `Bolji od GARCH?`: `DA **` p<0.05; `DA .` p<0.10; `-` nema znacajne razlike

**Sto ocekivati:** U punom istrazivanju M1 (LCC) i M2 (APL) NET modeli
znacajno pobjeduju GARCH (p~0.002). Na kracem uzorku signal moze biti slabiji.


In [ ]:
# Vizualizacija: QLIKE gubici kroz vrijeme
oos_idx = np.where(oos_mask)[0]
dates_oos = df["end_date"].iloc[oos_idx]

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(dates_oos, qlike(garch_hat[oos_mask], y_r[oos_mask]),
        "k-", lw=1.5, alpha=0.8, label="GARCH(1,1)")
for label, hat, color, ls in [
    ("Hist. mean", hist_mean_hat, "gray",       ":"),
    ("Last obs.",  last_obs_hat,  "purple",      "-."),
    ("NET M1",     net_m1_hat,    "steelblue",   "--"),
    ("NET M2",     net_m2_hat,    "darkorange",  "--"),
]:
    mask = oos_mask & ~np.isnan(hat)
    if mask.sum() > 0:
        ax.plot(df["end_date"].iloc[np.where(mask)[0]],
                qlike(hat[mask], y_r[mask]),
                ls, color=color, lw=1.2, alpha=0.9, label=label)
ax.set_ylabel("QLIKE gubitak (nizi = bolji)")
ax.set_title("OOS QLIKE gubitak: NET mreza vs. GARCH(1,1)")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("nb03_oos.png", dpi=120, bbox_inches="tight")
plt.show()


**Kako citati OOS graf:**

- Svaka tocka = QLIKE gubitak za jedan OOS prozor (crna = GARCH, plava = NET M1, narancasta = NET M2)
- Periodi gdje su plava/narancasta **ispod crne** → NET je bolji od GARCH u tom prozoru
- Periodi gdje su iznad → GARCH je bolji
- Volatilni periodi (GFC ~2008-2009, COVID ~2020) cesto daju najvece gubitke svim modelima
- Ako NET dosljednije ostaje ispod crne crte → sustavno bolja prognoza

> Ukupna prognozna prednos mjeri se **prosjecnim** QLIKE i DM testom (tablica iznad),
> a ne samo jednim periodom.


---

## Zadaci za studente

**1.** Koja mjera ima najjaci (po apsolutnoj vrijednosti) Spearman rho s vol_next?
   Je li predznak u ocekivanom smjeru?

**2.** Ponovite analizu za **M8** (eigenvector centralnost).
   Ocekujete li pozitivan ili negativan rho? Zasto?

**3.** Dodajte **M6** (apsorcijski omjer) u kondicionalni scatter plot (Sekcija 4).
   M6 je uvijek dostupan (nije kondicionalan) — je li rho u ocekivanom smjeru?

**4.** (Napredno) Usporedite rho na **W90 prozorima** (sample_metrics_W90.csv)
   vs. revizijskim prozorima. Je li veza jaca na jednom skupu?
   Sto to govori o izboru vremenskog prozora za analizu?
